# Loss-Grid Computer: Train Missing Checkpoints

Trains same-family checkpoint variants needed for the `torch.compile` amortization experiment.

This notebook:
- Mounts Drive and links `assets/` so checkpoints persist automatically
- Clones or reuses the repo from Drive
- Trains missing checkpoints via `training/train_missing_checkpoints.py`
- Verifies each saved checkpoint is loadable

Target runtime: CUDA GPU (T4 recommended). Set `SEEDS` and `WORKLOADS` in the configuration cell before running all.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Must match the layout used by functional_eval_colab.ipynb.
#   DRIVE_ROOT/
#     assets/            <- checkpoints and datasets live here
#     loss-grid-computer/ <- repo mirror (optional)
DRIVE_ROOT = '/content/drive/MyDrive/loss-grid-experiments'
DRIVE_ASSETS_ROOT = f'{DRIVE_ROOT}/assets'

## 2. Load repo and install dependencies

In [ ]:
import os
import subprocess
from pathlib import Path

if 'DRIVE_ROOT' not in globals():
    DRIVE_ROOT = '/content/drive/MyDrive/loss-grid-experiments'
    DRIVE_ASSETS_ROOT = f'{DRIVE_ROOT}/assets'

REPO_DIR = Path('/content/loss-grid-computer')
DRIVE_REPO_DIR = Path(DRIVE_ROOT) / 'loss-grid-computer'

if REPO_DIR.exists():
    print(f'Repo already present at {REPO_DIR}')
elif DRIVE_REPO_DIR.exists():
    print(f'Copying repo from Drive...')
    subprocess.check_call(['cp', '-r', str(DRIVE_REPO_DIR), str(REPO_DIR)])
    print(f'Repo copied to {REPO_DIR}')
else:
    print('Cloning repo from GitHub...')
    subprocess.check_call([
        'git', 'clone',
        'https://github.com/hotz99/loss-grid-computer.git',
        str(REPO_DIR),
    ])
    print(f'Repo cloned to {REPO_DIR}')

os.chdir(REPO_DIR)
print(f'Working directory: {Path.cwd()}')

In [ ]:
import sys
import subprocess

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch', 'torchvision',
    'scikit-learn', 'pandas', 'numpy',
])
print('Dependencies installed.')

## 3. Link assets from Drive

Symlinking `assets/` to the Drive folder means every checkpoint written by the
training script lands directly in Drive — no explicit copy step needed.

In [ ]:
import os
from pathlib import Path

assets_dst = Path('assets')
assets_src = Path(DRIVE_ASSETS_ROOT)

if not assets_src.exists():
    assets_src.mkdir(parents=True)
    print(f'Created Drive assets folder: {assets_src}')

if assets_dst.is_symlink():
    print(f'Assets symlink already present: {assets_dst} -> {os.readlink(assets_dst)}')
elif assets_dst.exists():
    print(f'Assets already present (not a symlink): {assets_dst.resolve()}')
else:
    assets_dst.symlink_to(assets_src)
    print(f'Symlinked: {assets_dst} -> {assets_src}')

print('Checkpoint files currently in assets/:')
for p in sorted(assets_dst.glob('*.pkl')):
    print(f'  {p.name}  ({p.stat().st_size / 1e3:.0f} KB)')

## 4. Verify CUDA runtime

In [ ]:
import os
import torch

assert torch.cuda.is_available(), 'No CUDA GPU found. Change Runtime > Change runtime type > GPU.'
gpu_name = torch.cuda.get_device_name(0)
props = torch.cuda.get_device_properties(0)
print(f'GPU  : {gpu_name}')
print(f'VRAM : {props.total_memory / 1e9:.1f} GB')
print(f'CPU  : {os.cpu_count()} cores')
print(f'Torch: {torch.__version__}')

## 5. Training configuration

Set the seeds and workloads to train here. `SEEDS` must be the three additional
seeds to complement the existing seed-0 checkpoints for each workload.
These match the seeds chosen for the `torch.compile` amortization experiment.

In [ ]:
# Seeds that complement the existing seed-0 checkpoints.
# Must produce N=4 total per workload when combined with seed-0.
SEEDS = [42, 99, 1337]

# Workloads to train. Set to None to train all three.
WORKLOADS = None  # or e.g. ['mnist_mlp', 'california_mlp', 'cifar10_row_gru']

DEVICE = 'cuda'

print(f'Seeds   : {SEEDS}')
print(f'Workloads: {WORKLOADS or "all"}')
print(f'Device  : {DEVICE}')

## 6. Dry run — verify plan before training

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, 'training/train_missing_checkpoints.py',
    '--dry-run',
    '--device', DEVICE,
    '--seeds', *[str(s) for s in SEEDS],
]
if WORKLOADS:
    for w in WORKLOADS:
        cmd += ['--workload', w]

result = subprocess.run(cmd, capture_output=False, text=True)
print('Exit code:', result.returncode)

## 7. Train missing checkpoints

Runs each missing `(workload, seed)` pair sequentially. Checkpoints are written
directly to `assets/` (symlinked to Drive) so they persist if the session
disconnects between runs.

Expected wall time on T4 (rough):
- `california_mlp`: ~1 min per seed (tabular, 200 epochs)
- `mnist_mlp`: ~3 min per seed (200 epochs)
- `cifar10_row_gru`: ~15 min per seed (80 epochs, CIFAR-10)

Total: ~57 min for all 9 checkpoints.

In [ ]:
import subprocess
import sys

cmd = [
    sys.executable, 'training/train_missing_checkpoints.py',
    '--device', DEVICE,
    '--seeds', *[str(s) for s in SEEDS],
]
if WORKLOADS:
    for w in WORKLOADS:
        cmd += ['--workload', w]

result = subprocess.run(cmd)
print('\nExit code:', result.returncode)
if result.returncode != 0:
    raise RuntimeError('One or more training runs failed — check output above.')

## 8. Verify saved checkpoints

In [ ]:
import torch
from pathlib import Path

expected_names = [
    f'{prefix}-{seed}.pkl'
    for prefix in ('california-mlp', 'mnist-mlp', 'cifar10-row-gru')
    for seed in SEEDS
]

assets = Path('assets')
print(f'{'File':<35} {'Size (KB)':>10}  Status')
print('-' * 60)
missing = []
for name in expected_names:
    p = assets / name
    if not p.exists():
        print(f'{name:<35} {'':>10}  MISSING')
        missing.append(name)
        continue
    try:
        state = torch.load(p, map_location='cpu', weights_only=True)
        n_tensors = len(state)
        size_kb = p.stat().st_size / 1e3
        print(f'{name:<35} {size_kb:>10.0f}  ok ({n_tensors} tensors)')
    except Exception as exc:
        print(f'{name:<35} {'':>10}  ERROR: {exc}')
        missing.append(name)

if missing:
    print(f'\nWARNING: {len(missing)} checkpoint(s) missing or unloadable.')
else:
    print(f'\nAll {len(expected_names)} checkpoints verified.')